# Assignment 3: Two Averages, One Argument — Descriptive Analytics (`2_work.ipynb`)
**Dataset:** `teaching_prices_clean.csv` (Illustrative teaching data — not actual PBS observations)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('teaching_prices_clean.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['item', 'city', 'date']).reset_index(drop=True)

print(f"Rows: {len(df)}, Columns: {list(df.columns)}")
df.head(8)

## 1. Item-Level Mean, Median, Spread & First-to-Last Percentage Change

In [2]:
rows = []
for item, grp in df.groupby('item'):
    mean_val = grp['price'].mean()
    median_val = grp['price'].median()
    std_val = grp['price'].std()
    
    c1 = grp[grp['city'] == 'City 1'].sort_values('date')
    c2 = grp[grp['city'] == 'City 2'].sort_values('date')
    pct_c1 = (c1['price'].iloc[-1] - c1['price'].iloc[0]) / c1['price'].iloc[0] * 100
    pct_c2 = (c2['price'].iloc[-1] - c2['price'].iloc[0]) / c2['price'].iloc[0] * 100
    
    by_date = grp.groupby('date')['price'].mean().sort_index()
    pct_combined = (by_date.iloc[-1] - by_date.iloc[0]) / by_date.iloc[0] * 100
    
    rows.append({
        'Item': item,
        'N': len(grp),
        'Mean': round(mean_val, 2),
        'Median': round(median_val, 2),
        'Mean_Minus_Median': round(mean_val - median_val, 2),
        'Std_Dev': round(std_val, 2),
        'Min': grp['price'].min(),
        'Max': grp['price'].max(),
        'City1_Pct_Change': round(pct_c1, 2),
        'City2_Pct_Change': round(pct_c2, 2),
        'Combined_Pct_Change': round(pct_combined, 2)
    })

stats_df = pd.DataFrame(rows)
stats_df

## 2. City-Level Breakdown (First vs. Last Date)

In [3]:
city_rows = []
for (item, city), grp in df.groupby(['item', 'city']):
    grp = grp.sort_values('date')
    first_p = grp['price'].iloc[0]
    last_p = grp['price'].iloc[-1]
    city_rows.append({
        'Item': item,
        'City': city,
        'Mean': round(grp['price'].mean(), 2),
        'Median': round(grp['price'].median(), 2),
        'First_Price (2026-01-01)': first_p,
        'Last_Price (2026-03-19)': last_p,
        'Pct_Change': round((last_p - first_p) / first_p * 100, 2)
    })

pd.DataFrame(city_rows)

## 3. Week 2 Flagged-Value Sensitivity Analysis (Staple A)

In [4]:
staple_clean = df[df['item'] == 'Staple A'].copy()
staple_flagged = staple_clean.copy()

# Flagged value: 10x decimal-shift recording error on 2026-03-19 City 1 (114.5 -> 1145.0)
mask = (staple_flagged['date'] == '2026-03-19') & (staple_flagged['city'] == 'City 1')
staple_flagged.loc[mask, 'price'] = 1145.0

comparison = pd.DataFrame([
    {
        'Dataset': 'With Flagged Value (1145.0 included)',
        'Mean (PKR/kg)': round(staple_flagged['price'].mean(), 2),
        'Median (PKR/kg)': round(staple_flagged['price'].median(), 2),
        'Mean - Median': round(staple_flagged['price'].mean() - staple_flagged['price'].median(), 2)
    },
    {
        'Dataset': 'Without Flagged Value (Cleaned to 114.5)',
        'Mean (PKR/kg)': round(staple_clean['price'].mean(), 2),
        'Median (PKR/kg)': round(staple_clean['price'].median(), 2),
        'Mean - Median': round(staple_clean['price'].mean() - staple_clean['price'].median(), 2)
    }
])
comparison

## 4. Visual Verification: Weekly Price Trajectories & Mean vs. Median

In [5]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
items = ['Staple A', 'Oil B', 'Pulse C', 'Sweetener D']

for ax, item in zip(axes.flat, items):
    sub = df[df['item'] == item]
    for city, c_grp in sub.groupby('city'):
        ax.plot(c_grp['date'], c_grp['price'], marker='o', label=city)
    ax.axhline(sub['price'].mean(), color='crimson', linestyle='--', linewidth=1.2, label=f"Mean ({sub['price'].mean():.2f})")
    ax.axhline(sub['price'].median(), color='teal', linestyle=':', linewidth=1.5, label=f"Median ({sub['price'].median():.2f})")
    ax.set_title(item)
    ax.set_ylabel('PKR per kg')
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Illustrative Teaching Prices: Weekly Trend, Mean vs. Median', fontsize=13)
plt.tight_layout()
plt.show()